In [7]:
from google.colab import files

uploaded = files.upload()

Saving questions_with_plans_retrieved.json to questions_with_plans_retrieved (2).json


In [8]:
# ════════════════════════════════════════════════════════════
#  CODE 3: Text → Plan+SQL  |  Evaluate Plan F1 + SQL F1
#  Upload: lora_adapter_combined.zip, questions_with_plans_(2).json
# ════════════════════════════════════════════════════════════

!pip install -q transformers==4.40.0 peft==0.10.0 accelerate==0.29.3

import json, zipfile, os, re, gc, time
from collections import Counter
import torch
from transformers import T5ForConditionalGeneration, AutoTokenizer
from peft import PeftModel

# ── Config ────────────────────────────────────────────────────
ADAPTER_ZIP  = "./lora_adapter (1).zip"
ADAPTER_DIR  = "./lora_adapter_combined"
DATA_PATH    = "./questions_with_plans_retrieved.json"
OUTPUT_PATH  = "./code3_combined_results.json"
SQL_SEP      = "[SQL]"
MAX_INPUT    = 512
MAX_TARGET   = 512
N_SAMPLES    = 101

# ── Unzip ─────────────────────────────────────────────────────
if not os.path.exists(ADAPTER_DIR):
    with zipfile.ZipFile(ADAPTER_ZIP, 'r') as z:
        z.extractall(ADAPTER_DIR)
    print("Extracted to:", ADAPTER_DIR)

# ── Load model ────────────────────────────────────────────────
MODEL_NAME = "google/flan-t5-xl"
dtype = torch.bfloat16 if torch.cuda.is_available() and \
        torch.cuda.get_device_capability()[0] >= 8 else torch.float16

print(f"Loading base model ({dtype})...")
base_model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype=dtype, device_map="auto",
)
print("Loading combined LoRA...")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR, is_trainable=False)
model.eval()
model.config.use_cache = True
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
print("Model ready!")

# ── Load data ─────────────────────────────────────────────────
with open(DATA_PATH, encoding="utf-8") as f:
    data = json.load(f)
samples = data[:N_SAMPLES]
print(f"Loaded {len(samples)} samples")

# ── Generate ──────────────────────────────────────────────────
def generate(input_text):
    inp = tokenizer(input_text, return_tensors="pt",
                    max_length=MAX_INPUT, truncation=True)
    inp = {k: v.to(model.device) for k, v in inp.items()}
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=MAX_TARGET,
                             num_beams=4, early_stopping=True, length_penalty=1.0)
    return tokenizer.decode(out[0], skip_special_tokens=True)

def split_output(text):
    if SQL_SEP in text:
        plan_part, sql_part = text.split(SQL_SEP, 1)
        return plan_part.strip(), sql_part.strip()
    return "", text.strip()

results = []
start = time.time()
for i, sample in enumerate(samples):
    pred_text = generate(sample["input"])
    pred_plan, pred_sql = split_output(pred_text)
    results.append({
        "idx"       : i,
        "input"     : sample["input"],
        "gold_plan" : sample["output"],
        "gold_sql"  : sample["sql"],
        "pred_plan" : pred_plan,
        "pred_sql"  : pred_sql,
        "pred_raw"  : pred_text,
        "db_id"     : sample.get("db_id", ""),
    })
    if (i + 1) % 25 == 0:
        elapsed = time.time() - start
        eta = elapsed / (i + 1) * (N_SAMPLES - i - 1)
        print(f"  [{i+1}/{N_SAMPLES}]  elapsed: {elapsed/60:.1f} min  ETA: {eta/60:.1f} min")

# ── Token F1 ──────────────────────────────────────────────────
def tokenize(text):
    return re.findall(r'\w+', text.lower())

def token_f1(pred, gold):
    pt = tokenize(pred); gt = tokenize(gold)
    if len(pt) == 0 and len(gt) == 0: return 1.0
    if len(pt) == 0 or  len(gt) == 0: return 0.0
    overlap   = sum((Counter(pt) & Counter(gt)).values())
    precision = overlap / len(pt)
    recall    = overlap / len(gt)
    if precision + recall == 0: return 0.0
    return 2 * precision * recall / (precision + recall)

plan_f1_scores = []
sql_f1_scores  = []
for r in results:
    pf1 = token_f1(r["pred_plan"], r["gold_plan"])
    sf1 = token_f1(r["pred_sql"],  r["gold_sql"])
    plan_f1_scores.append(pf1)
    sql_f1_scores.append(sf1)
    r["plan_f1"] = round(pf1, 4)
    r["sql_f1"]  = round(sf1, 4)

avg_plan_f1 = sum(plan_f1_scores) / len(plan_f1_scores)
avg_sql_f1  = sum(sql_f1_scores)  / len(sql_f1_scores)

print("\n══════════════════════════════════════════")
print(f"   CODE 3: COMBINED EVAL (n={N_SAMPLES})")
print("══════════════════════════════════════════")
print(f"  Plan Token F1 : {avg_plan_f1*100:.2f}%")
print(f"  SQL  Token F1 : {avg_sql_f1*100:.2f}%")
print("══════════════════════════════════════════")

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"Saved → {OUTPUT_PATH}")

print("\n── Samples ──")
for i in [0, 10, 50]:
    r = results[i]
    print(f"\n[{i}] Plan F1={r['plan_f1']:.3f}  SQL F1={r['sql_f1']:.3f}")
    print(f"  INPUT     : {r['input'][:80]}...")
    print(f"  GOLD PLAN : {r['gold_plan']}")
    print(f"  PRED PLAN : {r['pred_plan']}")
    print(f"  GOLD SQL  : {r['gold_sql']}")
    print(f"  PRED SQL  : {r['pred_sql']}")

Loading base model (torch.float16)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading combined LoRA...
Model ready!
Loaded 101 samples
  [25/101]  elapsed: 2.8 min  ETA: 8.4 min
  [50/101]  elapsed: 8.3 min  ETA: 8.4 min
  [75/101]  elapsed: 11.7 min  ETA: 4.1 min
  [100/101]  elapsed: 17.9 min  ETA: 0.2 min

══════════════════════════════════════════
   CODE 3: COMBINED EVAL (n=101)
══════════════════════════════════════════
  Plan Token F1 : 84.95%
  SQL  Token F1 : 84.68%
══════════════════════════════════════════
Saved → ./code3_combined_results.json

── Samples ──

[0] Plan F1=1.000  SQL F1=1.000
  INPUT     : question: How many products are there? | schema: product ( Product_ID [PK], Cate...
  GOLD PLAN : step1: SCAN | table: product || step2: AGGREGATE | Scalar aggregate (no GROUP BY)  ->  compute COUNT(*) || step3: PROJECT | columns: COUNT(*)
  PRED PLAN : step1: SCAN | table: product || step2: AGGREGATE | Scalar aggregate (no GROUP BY) -> compute COUNT(*) || step3: PROJECT | columns: COUNT(*)
  GOLD SQL  : SELECT count(*) FROM product
  PRED SQL  : SELE